<a href="https://colab.research.google.com/github/Elisabet-CueliAlba-ECA/out-of-sample-hierarchical-portfolios-hrp-hew/blob/main/04_M%C3%A9tricas_y_an%C3%A1lisis_vf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Introducción

En este notebook se comparan las distintas estrategias de construcción de portfolios estudiadas sobre distintos universos de activos usando métricas de rendimiento ajustado al riesgo (ASR, CEQ, MDD, TO, SSWP) y un test estadístico formal (MCS).

#Instalación e importación

In [ ]:

# 1. INSTALACIÓN DE LIBRERÍAS

import sys
import subprocess

# Intentamos importar arch de forma segura e instalarlo en caliente si no existe.
try:
    import arch
    from arch.bootstrap import MCS
except ImportError:
    print("[ENTORNO] Instalando la librería 'arch' para el MCS...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "arch"])
    import arch
    from arch.bootstrap import MCS

# 2. IMPORTACIÓN DE LIBRERÍAS

import os
import zipfile
import numpy as np
import pandas as pd
from scipy.stats import skew, kurtosis


In [ ]:
# 3. PARÁMETROS GENERALES Y CONFIGURACIÓN

from google.colab import drive

drive.mount('/content/drive')


BASE_PATH = "/content/drive/Othercomputers/Mi portátil (1)/0 Post lenovo/Año 4/TFG/Code/Files for code/Rolling window"

# Parámetros analíticos
PARAMS = {
    "rf": 0.0,                        # Tasa libre de riesgo
    "gamma": 1.0,                     # Aversión al riesgo para el CEQ
    "annualization_factor": 252,      # Días bursátiles al año

    # Parámetros formales del Model Confidence Set (MCS)
    "CONF_LEVELS": [0.20, 0.70],      # Niveles de confianza requeridos (se traducen a alpha = 1 - conf)
    "mcs_reps": 1000,                 # Repeticiones bootstrap
    "mcs_block_size": None,           # Tamaño de bloque (None para auto-detección)
    "mcs_method": "R"                 # Estadístico: 'R' (max T) o 'SQ' (suma de cuadrados)
}

CONJUNTOS = ["sectores", "multiassets", "subindiv_ward"]

ESTRATEGIAS_ORDEN = [
    "HRP_SL", "HEW_SL", "HEW_CL", "HEW_AL", "HEW_Ward", "HEW_DBHT",
    "1_N", "IVP", "MV", "ERC"
]


Mounted at /content/drive


In [ ]:

# DEFINICIÓN DE RUTA BASE

print(f"[ENTORNO] Ruta base definida para lecturas: {BASE_PATH}")
if not os.path.exists(BASE_PATH):
    print(f"[!] ATENCIÓN: La ruta base {BASE_PATH} no existe. Por favor, revísala.")

# CARGA DE ARCHIVOS

def load_returns(conjunto):
    """Carga los retornos OOS diarios desde la ruta base."""
    path = os.path.join(BASE_PATH, f"ret_oos_{conjunto}.csv")
    if os.path.exists(path):
        return pd.read_csv(path, index_col='fecha', parse_dates=True)
    else:
        print(f"[!] No se encontró el archivo de retornos para {conjunto}: {path}")
        return None


[ENTORNO] Ruta base definida para lecturas: /content/drive/Othercomputers/Mi portátil (1)/0 Post lenovo/Año 4/TFG/Code/Files for code/Rolling window


In [ ]:
# PREPARACIÓN DE PESOS

def build_weight_matrices(conjunto):
    """
    Convierte el almacenamiento largo o empaquetado en diccionarios
    estandarizados de matrices (fecha x activo) por estrategia.
    """
    w_dict = {}

    if conjunto in ["sectores", "multiassets"]:
        path = os.path.join(BASE_PATH, f"pesos_oos_{conjunto}.csv")
        if os.path.exists(path):
            df_long = pd.read_csv(path, parse_dates=['fecha'])
            for est in df_long['estrategia'].unique():
                df_est = df_long[df_long['estrategia'] == est]
                df_pivot = df_est.pivot(index='fecha', columns='activo', values='peso')
                w_dict[est] = df_pivot
        else:
            print(f"[!] Pesos no encontrados para {conjunto}: {path}")

    elif conjunto == "subindiv_ward":
        zip_path = os.path.join(BASE_PATH, "indivassets_backtest_exports.zip")
        if os.path.exists(zip_path):
            with zipfile.ZipFile(zip_path, 'r') as z:
                # Filtrar solo archivos de pesos
                csv_files = [f for f in z.namelist() if f.startswith('pesos_oos_indivassets_') and f.endswith('.csv')]
                if not csv_files:
                    print(f"[!] El ZIP de {conjunto} no contiene archivos de pesos reconocibles.")
                for f in csv_files:
                    est = f.replace('pesos_oos_indivassets_', '').replace('.csv', '')
                    with z.open(f) as csv_file:
                        df_pivot = pd.read_csv(csv_file, index_col='fecha', parse_dates=True)
                        w_dict[est] = df_pivot
        else:
            print(f"[!] ZIP de pesos no encontrado para {conjunto}: {zip_path}")

    return w_dict


In [ ]:
# 8. MÉTRICAS
# Las métricas se calculan con datos diarios y se anualizan explícitamente donde procede.

def calc_ASR(ret_series):
    """
    Adjusted Sharpe Ratio (ASR).
    SR = (mu - rf) / sigma
    ASR = SR * [1 + (mu3 / 6) * SR - ((mu4 - 3) / 24) * SR^2]
    """
    ret_clean = ret_series.dropna()
    if len(ret_clean) < 2: return np.nan

    mu = ret_clean.mean()
    sigma = ret_clean.std(ddof=1)
    if sigma == 0 or pd.isna(sigma): return np.nan

    SR = (mu - PARAMS["rf"]) / sigma
    mu3 = skew(ret_clean)
    mu4 = kurtosis(ret_clean, fisher=False) # Pearson kurtosis (normal=3)

    ASR_daily = SR * (1 + (mu3 / 6) * SR - ((mu4 - 3) / 24) * (SR ** 2))
    return ASR_daily * np.sqrt(PARAMS["annualization_factor"])

def calc_CEQ(ret_series):
    """
    Certainty Equivalent (CEQ).
    CEQ = (mu - rf) - (gamma / 2) * sigma^2
    """
    ret_clean = ret_series.dropna()
    if len(ret_clean) < 2: return np.nan

    mu = ret_clean.mean()
    sigma = ret_clean.std(ddof=1)

    CEQ_daily = (mu - PARAMS["rf"]) - (PARAMS["gamma"] / 2.0) * (sigma ** 2)
    return CEQ_daily * PARAMS["annualization_factor"]

def calc_MDD(ret_series):
    """
    Maximum Drawdown (MDD) absoluto calculado sobre la curva de riqueza empírica OOS.
    """
    ret_clean = ret_series.dropna()
    if len(ret_clean) == 0: return np.nan

    wealth = (1 + ret_clean).cumprod()
    peak = wealth.cummax()
    drawdown = (wealth - peak) / peak
    return abs(drawdown.min())

def calc_TO(w_matrix):
    """
    Turnover (TO). Promedio temporal de la suma de cambios absolutos de pesos sobre activos.
    """
    if w_matrix.empty: return np.nan
    w_diff = w_matrix.diff().abs()
    return w_diff.sum(axis=1).mean()

def calc_SSPW(w_matrix):
    """
    Sum of Squared Portfolio Weights (SSPW). Promedio temporal de la suma de pesos al cuadrado.
    """
    if w_matrix.empty: return np.nan
    w_sq = w_matrix ** 2
    return w_sq.sum(axis=1).mean()


# TABLAS RESUMEN

def build_metrics_table(ret_df, w_dict):
    """Construye la tabla de métricas finales blindando el orden oficial de estrategias."""
    metrics = []

    for est in ESTRATEGIAS_ORDEN:
        if est not in ret_df.columns:
            continue

        ret_s = ret_df[est]

        # Presentación en porcentaje para CEQ, MDD y TO
        ceq_val = calc_CEQ(ret_s)
        mdd_val = calc_MDD(ret_s)

        row = {
            "estrategia": est,
            "ASR": calc_ASR(ret_s),
            "CEQ": ceq_val * 100 if pd.notna(ceq_val) else np.nan,
            "MDD": mdd_val * 100 if pd.notna(mdd_val) else np.nan
        }

        if est in w_dict and not w_dict[est].empty:
            to_val = calc_TO(w_dict[est])
            row["TO"] = to_val * 100 if pd.notna(to_val) else np.nan
            row["SSPW"] = calc_SSPW(w_dict[est]) # SSPW no se multiplica por 100

        else:
            row["TO"] = np.nan
            row["SSPW"] = np.nan

        metrics.append(row)

    return pd.DataFrame(metrics).set_index("estrategia")


In [ ]:
# MCS

def run_arch_mcs(losses_df, alpha, prefix):
    """
    Ejecuta el MCS formal utilizando el objeto arch.bootstrap.MCS
    y extrae explícitamente los modelos incluidos y excluidos.
    Devuelve las columnas renombradas con el sufijo provisto (ej: _ret_20).
    """
    # Excluir fechas incompletas para evitar fallos en el block bootstrap
    L_clean = losses_df.dropna(axis=0, how='any')
    res = pd.DataFrame(index=losses_df.columns)


    if L_clean.empty or L_clean.shape[1] < 2:
        print(f"  [!] Datos insuficientes o menos de 2 modelos para ejecutar MCS ({prefix}).")
        res[f'mcs_{prefix}'] = np.nan
        res[f'pvalue_{prefix}'] = np.nan
        res[f'status_{prefix}'] = "Error"
        res[f'in_mcs_{prefix}'] = False
        res[f'confidence_level_{prefix}'] = 1 - alpha
        res[f'loss_type_{prefix}'] = prefix.split("_")[0]
        return res

    print(f"  -> Ejecutando arch.bootstrap.MCS ({prefix}) con {L_clean.shape[1]} modelos...")
    mcs = MCS(
        L_clean,
        size=alpha,
        reps=PARAMS["mcs_reps"],
        block_size=PARAMS["mcs_block_size"],
        method=PARAMS["mcs_method"]
    )
    mcs.compute()

    # Extraemos propiedades nativas de la librería
    pvals = mcs.pvalues
    included_models = list(mcs.included)

    # Identificar columna del p-value dinámicamente
    pval_col = "Pvalue" if "Pvalue" in pvals.columns else (pvals.select_dtypes(include=np.number).columns[0] if not pvals.empty else None)

    # Construcción de la tabla de salida clara y legible
    # Extraemos Pvalue base
    pvals_serie = np.full(len(losses_df.columns), np.nan)
    res_temp = pd.DataFrame(index=losses_df.columns)
    if pval_col is not None:
        res_temp['Pvalue'] = pvals[pval_col]
    else:
        res_temp['Pvalue'] = np.nan

    status_list = ["Included" if m in included_models else "Excluded" for m in res.index]
    in_mcs_list = [s == "Included" for s in status_list]

    # Asignamos al formato final
    res[f'mcs_{prefix}'] = in_mcs_list
    res[f'pvalue_{prefix}'] = res_temp['Pvalue']
    res[f'status_{prefix}'] = status_list
    res[f'in_mcs_{prefix}'] = in_mcs_list
    res[f'confidence_level_{prefix}'] = 1 - alpha
    res[f'loss_type_{prefix}'] = prefix.split("_")[0]

    return res


In [ ]:

# EXPORTACIÓN FINAL Y DESCARGA AUTOMÁTICA

def ejecutar_analisis_conjunto(conjunto_name):
    """
    Flujo central que procesa única y exclusivamente el conjunto solicitado,
    evitando arrastrar en memoria la carga de universos no requeridos.
    """
    out_dir = "/content"
    os.makedirs(out_dir, exist_ok=True)
    archivos_generados = []

    print(f"\n" + "="*50)
    print(f"PROCESANDO CONJUNTO EXCLUSIVO: {conjunto_name.upper()}")
    print("="*50)

    # Carga de retornos
    ret_df = load_returns(conjunto_name)
    if ret_df is None or ret_df.empty:
        print(f"  [!] Abortando ejecución: faltan retornos para {conjunto_name}.")
        return

    # Carga de pesos
    w_dict = build_weight_matrices(conjunto_name)

    # 1. Tabla de Métricas
    df_metricas = build_metrics_table(ret_df, w_dict)
    f_metricas = os.path.join(out_dir, f"tabla_metricas_{conjunto_name}.csv")
    df_metricas.to_csv(f_metricas)
    archivos_generados.append(f_metricas)
    print(f"  -> Tabla de métricas exportada a: {f_metricas}")

    # 2. MCS Consolidado (Retornos y CEQ por Niveles de Confianza)
    # Pérdida 1: Retornos diarios
    losses_ret = -ret_df

    # Pérdida 2: CEQ diario -> u_t = (r_t - rf) - (gamma / 2) * (r_t - rf)^2
    rf = PARAMS["rf"]
    gamma = PARAMS["gamma"]
    u_t = (ret_df - rf) - (gamma / 2.0) * (ret_df - rf)**2
    losses_ceq = -u_t

    mcs_dfs = []

    # Construir en el orden exacto solicitado: ret_20, ret_70, ceq_20, ceq_70
    conf_levels = PARAMS.get("CONF_LEVELS", [0.20, 0.70])

    for conf in conf_levels:
        alpha = 1.0 - conf
        conf_str = str(int(conf * 100))
        mcs_dfs.append(run_arch_mcs(losses_ret, alpha=alpha, prefix=f"ret_{conf_str}"))

    for conf in conf_levels:
        alpha = 1.0 - conf
        conf_str = str(int(conf * 100))
        mcs_dfs.append(run_arch_mcs(losses_ceq, alpha=alpha, prefix=f"ceq_{conf_str}"))

    # Unificar todas las columnas del MCS
    df_mcs_consolidado = pd.concat(mcs_dfs, axis=1)
    df_mcs_consolidado.index.name = "estrategia"

    f_mcs = os.path.join(out_dir, f"mcs_{conjunto_name}.csv")
    df_mcs_consolidado.to_csv(f_mcs)
    archivos_generados.append(f_mcs)
    print(f"  -> Resultado MCS consolidado exportado a: {f_mcs}")

    # Empaquetado final (ZIP) exclusivo para el conjunto procesado
    zip_final = None
    if len(archivos_generados) > 0:
        zip_final = os.path.join(out_dir, f"metricas_y_mcs_{conjunto_name}.zip")
        print(f"\n[EXPORTACIÓN] Empaquetando CSVs en {zip_final}...")
        with zipfile.ZipFile(zip_final, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for f in archivos_generados:
                zipf.write(f, arcname=os.path.basename(f))

    # Descarga automática focalizada EXCLUSIVAMENTE en el ZIP
    print("\n[DESCARGA] Lanzando descarga automática del ZIP desde Colab...")
    try:
        from google.colab import files
        if zip_final and os.path.exists(zip_final):
            files.download(zip_final)
    except ImportError:
        print("[DESCARGA] Entorno local detectado. No se llama a files.download().")
        print(f"Los resultados están en: {out_dir}")


#Extracción

##Sectores

In [ ]:

# =====================================================================
# EJECUCIÓN REAL DEL FLUJO
# =====================================================================

RUN_SET = "sectores"                # Opciones válidas: "sectores", "multiassets", "subindiv_ward"


if __name__ == "__main__":
    if RUN_SET in CONJUNTOS:
        ejecutar_analisis_conjunto(RUN_SET)
    else:
        print(f"[!] Error: El conjunto '{RUN_SET}' no es válido. Usa uno de {CONJUNTOS}.")



PROCESANDO CONJUNTO EXCLUSIVO: SECTORES
  -> Tabla de métricas exportada a: /content/tabla_metricas_sectores.csv
  -> Ejecutando arch.bootstrap.MCS (ret_20) con 10 modelos...
  -> Ejecutando arch.bootstrap.MCS (ret_70) con 10 modelos...
  -> Ejecutando arch.bootstrap.MCS (ceq_20) con 10 modelos...
  -> Ejecutando arch.bootstrap.MCS (ceq_70) con 10 modelos...
  -> Resultado MCS consolidado exportado a: /content/mcs_sectores.csv

[EXPORTACIÓN] Empaquetando CSVs en /content/metricas_y_mcs_sectores.zip...

[DESCARGA] Lanzando descarga automática del ZIP desde Colab...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

##Multiassets

In [ ]:
# =====================================================================
# EJECUCIÓN REAL DEL FLUJO
# =====================================================================

RUN_SET = "multiassets"                # Opciones válidas: "sectores", "multiassets", "subindiv_ward"


if __name__ == "__main__":
    if RUN_SET in CONJUNTOS:
        ejecutar_analisis_conjunto(RUN_SET)
    else:
        print(f"[!] Error: El conjunto '{RUN_SET}' no es válido. Usa uno de {CONJUNTOS}.")



PROCESANDO CONJUNTO EXCLUSIVO: MULTIASSETS
  -> Tabla de métricas exportada a: /content/tabla_metricas_multiassets.csv
  -> Ejecutando arch.bootstrap.MCS (ret_20) con 10 modelos...
  -> Ejecutando arch.bootstrap.MCS (ret_70) con 10 modelos...
  -> Ejecutando arch.bootstrap.MCS (ceq_20) con 10 modelos...
  -> Ejecutando arch.bootstrap.MCS (ceq_70) con 10 modelos...
  -> Resultado MCS consolidado exportado a: /content/mcs_multiassets.csv

[EXPORTACIÓN] Empaquetando CSVs en /content/metricas_y_mcs_multiassets.zip...

[DESCARGA] Lanzando descarga automática del ZIP desde Colab...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

##Indivassets

In [ ]:
# =====================================================================
# EJECUCIÓN REAL DEL FLUJO
# =====================================================================

RUN_SET = "subindiv_ward"                # Opciones válidas: "sectores", "multiassets", "subindiv_ward"


if __name__ == "__main__":
    if RUN_SET in CONJUNTOS:
        ejecutar_analisis_conjunto(RUN_SET)
    else:
        print(f"[!] Error: El conjunto '{RUN_SET}' no es válido. Usa uno de {CONJUNTOS}.")



PROCESANDO CONJUNTO EXCLUSIVO: SUBINDIV_WARD
  -> Tabla de métricas exportada a: /content/tabla_metricas_subindiv_ward.csv
  -> Ejecutando arch.bootstrap.MCS (ret_20) con 10 modelos...
  -> Ejecutando arch.bootstrap.MCS (ret_70) con 10 modelos...
  -> Ejecutando arch.bootstrap.MCS (ceq_20) con 10 modelos...
  -> Ejecutando arch.bootstrap.MCS (ceq_70) con 10 modelos...
  -> Resultado MCS consolidado exportado a: /content/mcs_subindiv_ward.csv

[EXPORTACIÓN] Empaquetando CSVs en /content/metricas_y_mcs_subindiv_ward.zip...

[DESCARGA] Lanzando descarga automática del ZIP desde Colab...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>